# End-to-End Retail Cohort & Retention Analysis

**Project Goal:** Move beyond surface-level "vanity metrics" (total revenue, total orders) and answer the fundamental customer lifecycle question:  
> *How many customers actually come back after their first purchase — and when do they stop?*

**Dataset:** UCI Online Retail Dataset (Dec 2010 – Dec 2011)  
**Technologies:** Python (Pandas, NumPy), SQL Window Functions (via `pandasql`), Power BI  

---

## Pipeline Overview

| Phase | What happens |
|-------|-------------|
| **A — Structural Audit** | Load raw data, quantify missing IDs, returns, and zero-price anomalies |
| **B — Data Cleaning** | Remove un-trackable rows and standardise timestamps |
| **C — SQL Cohort Logic** | Use Window Functions to assign each customer a birth month and cohort age index |
| **D — Retention Matrix** | Pivot and normalise counts into a percentage-based heatmap |
| **E — BI Export** | Save enriched CSV for Power BI dashboard consumption |


---
## Phase A — Structural Data Audit

Before writing a single line of analysis, we audit the 541,909 raw records for three critical data-quality issues:

| Issue | Why it matters |
|-------|---------------|
| **Missing `CustomerID`** | Retention tracking requires a unique user identifier. Anonymous rows are untraceable. |
| **Negative `Quantity`** (returns) | Returns inflate transaction counts and skew average-order-value calculations. |
| **Zero/Negative `UnitPrice`** | Free or error-priced items distort revenue metrics. |

Findings from the audit:
- **24.93 %** of rows (135,080) have no `CustomerID` → dropped entirely.
- **10,624** return transactions → filtered out.
- **2,517** zero-priced items → filtered out.


In [1]:
import pandas as pd
import numpy as np

# 1. Load the dataset directly from the UCI Source
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
print("Loading Dataset...")
df = pd.read_excel(url)

# 2. Structural Audit
print("--- DATA AUDIT REPORT ---")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

# Check for the 'Big Three' Issues
missing_ids = df['CustomerID'].isnull().sum()
negative_qty = (df['Quantity'] < 0).sum()
zero_price = (df['UnitPrice'] <= 0).sum()

print(f"Missing Customer IDs: {missing_ids} ({round(missing_ids/len(df)*100, 2)}%)")
print(f" Negative Quantities (Returns): {negative_qty}")
print(f" Unit Price <= 0: {zero_price}")

Loading Dataset...
--- DATA AUDIT REPORT ---
Total Rows: 541909
Total Columns: 8
Missing Customer IDs: 135080 (24.93%)
 Negative Quantities (Returns): 10624
 Unit Price <= 0: 2517


### Exploratory Data Audit — Schema & Distributions

We inspect three things to understand the data structure:
1. **`df.head()`** — Verify column names, date format, and that `CustomerID` is numeric.
2. **`df.info()`** — Confirm data types (is `InvoiceDate` already a `datetime`?).
3. **`df[...].describe()`** — Spot extreme outliers (min Quantity = −80,995 confirms mass cancellations).


In [2]:
# 1. The 'Head': See the first 5 rows to understand the structure
print("--- FIRST 5 ROWS ---")
display(df.head())

# 2. The 'Info': Check Data Types (Are dates actually dates?)
print("\n--- DATA TYPES & NULLS ---")
print(df.info())

# 3. The 'Describe': Statistical Audit (Find the 'Rich' vs 'Poor' customers)
print("\n--- STATISTICAL SUMMARY ---")
display(df[['Quantity', 'UnitPrice']].describe())

--- FIRST 5 ROWS ---


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom



--- DATA TYPES & NULLS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB
None

--- STATISTICAL SUMMARY ---


,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


---
## Phase B — Data Cleaning

Three targeted filters turn 541,909 raw rows into 397,884 analysis-ready rows:

1. **Drop anonymous transactions** — rows where `CustomerID` is null cannot be linked across visits.  
2. **Remove returns & freebies** — keep only `Quantity > 0` and `UnitPrice > 0` to ensure we count genuine sales only.  
3. **Normalise date granularity** — truncate `InvoiceDate` timestamps to month precision (`InvoiceMonth`).  
   - We care that a customer bought in *December 2010*, not at *08:26 AM* on the 1st.  
   - This is the foundation for all cohort grouping later.


In [3]:
# 1. Kick out the ghosts (No ID = No Entry)
df_clean = df.dropna(subset=['CustomerID'])

# 2. Only keep real sales (No returns, no freebies)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

# 3. Simplify the Date
# We don't care if they bought at 8:26 AM. We just care it was 'December 2010'.
df_clean['InvoiceMonth'] = df_clean['InvoiceDate'].dt.to_period('M').dt.to_timestamp()

print(f"Cleaned! We went from 541k rows down to {len(df_clean)} high-quality rows.")

Cleaned! We went from 541k rows down to 397884 high-quality rows.


---
## Phase C — Advanced SQL Cohort Logic (Window Functions)

This is the analytical core of the project. We use `pandasql` to run real SQL — including **Window Functions** — directly against the Pandas DataFrame.

### Why Window Functions instead of loops or merges?

A naive approach would `groupby(CustomerID).min()` and then merge back. Window Functions are cleaner:  
they compute the result *per row* without collapsing the DataFrame, so downstream joins are eliminated.

### The two-stage CTE logic

**Stage 1 — `cohort_lookup`:** Assign every transaction row its customer's birth month.
```sql
MIN(InvoiceMonth) OVER(PARTITION BY CustomerID) AS CohortMonth
```
> *For customer 12347, every one of their rows gets stamped with "Dec 2010" — the month of their very first order.*

**Stage 2 — `age_calculation`:** Compute how many months have elapsed since the birth month.
```sql
(strftime('%Y', InvoiceMonth) - strftime('%Y', CohortMonth)) * 12
+ (strftime('%m', InvoiceMonth) - strftime('%m', CohortMonth)) + 1 AS CohortIndex
```
> *CohortIndex = 1 means the customer transacted in their birth month. CohortIndex = 6 means they returned 5 months later.*

**Final SELECT:** Aggregate to unique customers per (CohortMonth, CohortIndex) pair — the raw input for the retention matrix.


In [4]:
!pip install pandasql
from pandasql import sqldf
pysqldf = lambda q: sqldf(q, globals())

# The 'Master' Cohort Query
query = """
WITH cohort_lookup AS (
    -- 1. Use a Window Function to find the 'Birth Month' for every row
    -- This 'partitions' the data by Customer so each person gets their own MIN date
    SELECT
        CustomerID,
        InvoiceMonth,
        MIN(InvoiceMonth) OVER(PARTITION BY CustomerID) as CohortMonth
    FROM df_clean
),
age_calculation AS (
    -- 2. Calculate the 'Age' (Index) by subtracting the dates
    -- We multiply years by 12 and add the month difference
    SELECT
        *,
        (strftime('%Y', InvoiceMonth) - strftime('%Y', CohortMonth)) * 12 +
        (strftime('%m', InvoiceMonth) - strftime('%m', CohortMonth)) + 1 AS CohortIndex
    FROM cohort_lookup
)
-- 3. Final Summary: Count unique customers for every Cohort and Age
SELECT
    CohortMonth,
    CohortIndex,
    COUNT(DISTINCT CustomerID) as TotalUsers
FROM age_calculation
GROUP BY 1, 2
ORDER BY 1, 2
"""

# Execute and view the first few rows
sql_cohort_results = pysqldf(query)
display(sql_cohort_results.head(15))

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=9d49bbaae02cf51c18a743fa7d4ac3bc491af3c43a8d64786f4e725bdcc5bb86
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


,CohortMonth,CohortIndex,TotalUsers
0,2010-12-01 00:00:00.000000,1,885
1,2010-12-01 00:00:00.000000,2,324
2,2010-12-01 00:00:00.000000,3,286
3,2010-12-01 00:00:00.000000,4,340
4,2010-12-01 00:00:00.000000,5,321
5,2010-12-01 00:00:00.000000,6,352
6,2010-12-01 00:00:00.000000,7,321
7,2010-12-01 00:00:00.000000,8,309
8,2010-12-01 00:00:00.000000,9,313
9,2010-12-01 00:00:00.000000,10,350


---
## Phase D — Building the Retention Heatmap

We convert the long-format SQL output into a percentage-based retention matrix.

### Step 1 — Pivot
Reshape from long → wide: rows = cohort birth month, columns = cohort age (1, 2, 3 … 13).

### Step 2 — Normalise
Divide every cell by the **Month 1** value (the cohort's starting size) to get a retention rate:
```
Retention% = (Users active in Month N) / (Users who joined in Month 1) × 100
```

### Reading the triangle
- **NaN cells** are expected — later cohorts haven't had time to reach older age indices yet.
- **Row = cohort** — "who joined in that month?"
- **Column = age** — "how many months since they joined?"
- **Value = % of original cohort still active**

### Key findings
| Observation | Evidence |
|-------------|----------|
| **Dec 2010 anniversary effect** | Month 12 retention spikes to **50.3 %** — holiday shoppers return a year later |
| **"Month 1" cliff** | Most cohorts fall from 100 % to ~15–22 % by Month 2 — high one-time buyer rate |
| **Stabilisation after Month 3** | Customers who survive past 3 months tend to stay; retention flattens at ~20–35 % |


In [8]:
# 1. Pivot the SQL results to create the 'Triangle' grid
# Rows = When they joined (Birthday)
# Columns = How many months they stayed (Age)
cohort_pivot = sql_cohort_results.pivot(index='CohortMonth',
                                        columns='CohortIndex',
                                        values='TotalUsers')

# 2. Calculate Percentages (%)
# We divide every month by the 'Month 1' (the total size of that group)
cohort_sizes = cohort_pivot.iloc[:,0]
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0).round(3) * 100

print("- FINAL RETENTION HEATMAP (%) -")
display(retention_matrix)

- FINAL RETENTION HEATMAP (%) -


CohortIndex,1,2,3,4,5,6,7,8,9,10,11,12,13
CohortMonth,,,,,,,,,,,,,
2010-12-01 00:00:00.000000,100.0,36.6,32.3,38.4,36.3,39.8,36.3,34.9,35.4,39.5,37.4,50.3,26.6
2011-01-01 00:00:00.000000,100.0,22.1,26.6,23.0,32.1,28.8,24.7,24.2,30.0,32.6,36.5,11.8,NaN
2011-02-01 00:00:00.000000,100.0,18.7,18.7,28.4,27.1,24.7,25.3,27.9,24.7,30.5,6.8,NaN,NaN
2011-03-01 00:00:00.000000,100.0,15.0,25.2,19.9,22.3,16.8,26.8,23.0,27.9,8.6,NaN,NaN,NaN
2011-04-01 00:00:00.000000,100.0,21.3,20.3,21.0,19.7,22.7,21.7,26.0,7.3,NaN,NaN,NaN,NaN
2011-05-01 00:00:00.000000,100.0,19.0,17.3,17.3,20.8,23.2,26.4,9.5,NaN,NaN,NaN,NaN,NaN
2011-06-01 00:00:00.000000,100.0,17.4,15.7,26.4,23.1,33.5,9.5,NaN,NaN,NaN,NaN,NaN,NaN
2011-07-01 00:00:00.000000,100.0,18.1,20.7,22.3,27.1,11.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-08-01 00:00:00.000000,100.0,20.7,24.9,24.3,12.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## Phase E — Data Modelling for Power BI Export

We now enrich the cleaned transaction-level DataFrame (`df_clean`) with the two cohort columns — `CohortMonth` and `CohortIndex` — so that every row carries its analytical context.

This approach means Power BI can build visuals **instantly** with simple field drags, without requiring heavy DAX calculations or multiple table relationships.

### Columns added to the final export

| Column | Type | Description |
|--------|------|-------------|
| `CohortMonth` | `datetime` | The customer's first-purchase month (their "birth month") |
| `CohortIndex` | `int` | Months elapsed since birth month (1 = first month, 2 = one month later, …) |


In [13]:
# 1. Run the SQL logic to create the missing columns
query = """
SELECT
    *,
    MIN(InvoiceMonth) OVER(PARTITION BY CustomerID) as CohortMonth,
    (strftime('%Y', InvoiceMonth) - strftime('%Y', MIN(InvoiceMonth) OVER(PARTITION BY CustomerID))) * 12 +
    (strftime('%m', InvoiceMonth) - strftime('%m', MIN(InvoiceMonth) OVER(PARTITION BY CustomerID))) + 1 AS CohortIndex
FROM df_clean
"""

# 2. Update your main dataframe
df_clean = pysqldf(query)

# 3. Verify the columns are there
print("--- NEW COLUMNS CHECK ---")
print(df_clean[['CustomerID', 'InvoiceMonth', 'CohortMonth', 'CohortIndex']].head())

--- NEW COLUMNS CHECK ---
   CustomerID                InvoiceMonth                 CohortMonth  \
0     12346.0  2011-01-01 00:00:00.000000  2011-01-01 00:00:00.000000   
1     12347.0  2010-12-01 00:00:00.000000  2010-12-01 00:00:00.000000   
2     12347.0  2010-12-01 00:00:00.000000  2010-12-01 00:00:00.000000   
3     12347.0  2010-12-01 00:00:00.000000  2010-12-01 00:00:00.000000   
4     12347.0  2010-12-01 00:00:00.000000  2010-12-01 00:00:00.000000   

   CohortIndex  
0            1  
1            1  
2            1  
3            1  
4            1  


### CSV Export — `Retail_Cohort_Data_Final.csv`

The enriched DataFrame is exported as a flat CSV. This file contains:
- All original cleaned transaction fields (InvoiceNo, StockCode, CustomerID, Country, etc.)
- Standardised `InvoiceMonth` timestamps
- `CohortMonth` — the customer's acquisition month
- `CohortIndex` — the customer's "age" in months at the time of each transaction

**Power BI Dashboard layers built on top of this file:**
- **Heatmap** — conditional formatting over the retention matrix
- **Country Slicers** — compare UK vs International retention curves
- **KPI Cards** — Average Monthly Retention % and Total Unique Customers per Cohort


In [14]:
# Save the master file for Power BI
# This includes the CustomerID, Birth Month, and Age Index for every transaction
df_clean.to_csv('Retail_Cohort_Data_Final.csv', index=False)

### File Path Verification

Confirming the CSV was saved successfully before handing off to Power BI.


In [11]:
import os

file_path = 'Retail_Cohort_Data_Final.csv'
if os.path.exists(file_path):
    print(f"The file '{file_path}' is located at: {os.path.abspath(file_path)}")
else:
    print(f"The file '{file_path}' was not found.")

The file 'Retail_Cohort_Data_Final.csv' is located at: /content/Retail_Cohort_Data_Final.csv
